In [1]:
import torch
import torchvision
from torchvision import datasets, transforms
from torch.utils.data import DataLoader

# 1. 定义数据预处理
# ToTensor 将图像转换为 [0.0, 1.0] 的张量
# Normalize 可选，通常用于加速训练收敛
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5))#(均值)，(标准差)
])


In [2]:
# 3. 载入训练集 (包含 50000 张图)
# train=True: torchvision 会自动找到并合并 data_batch_1 至 data_batch_5
ROOT_DIR = 'C:/Jupyter(Anaconda)/data'
CIFAR_dataset = torchvision.datasets.CIFAR10(
    root=ROOT_DIR,
    train=True,
    download=False,
    transform=transform
)

# --- 验证加载情况 ---
print(f"训练集大小: {len(CIFAR_dataset)}")

# 获取单个样本，验证是否返回 Tensor
image, label = CIFAR_dataset[0]
print(f"单图 Shape: {image.shape}, 标签: {label}")



训练集大小: 50000
单图 Shape: torch.Size([3, 32, 32]), 标签: 6


In [3]:
from torch.utils.data import Subset
# 直接切片（仅保留前 N 个，速度最快。⚠️需确保原数据已打乱顺序）
N = 15000  # 修改为你想要的训练样本数量
torch.manual_seed(42)#可复现
idx = torch.randperm(len(CIFAR_dataset))  # 生成随机排列索引

# 1. 包装为子集
train_subset = Subset(CIFAR_dataset, idx[:N])


X_train = torch.stack([img for img, _ in train_subset])
y_train = torch.tensor([lbl for _, lbl in train_subset])

print(f"X_train shape: {X_train.shape}")
print(f"y_train shape: {y_train.shape}")

X_train shape: torch.Size([15000, 3, 32, 32])
y_train shape: torch.Size([15000])


In [4]:
test_subset=Subset(CIFAR_dataset, idx[N:])
X_test=torch.stack([img for img, _ in test_subset])
y_test=torch.tensor([lbl for _, lbl in test_subset])

In [5]:
import torch.nn as nn
import torchvision.models as models

# 1. 定义设备
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

# 2. 加载模型并移至 GPU
model = models.resnet18(weights=models.ResNet18_Weights.DEFAULT)
model.fc = nn.Identity()  # 移除全连接层
model = model.to(device)
model.eval()

# 3. 定义归一化参数并移至 GPU
resnet_mean = torch.tensor([0.485, 0.456, 0.406]).view(3, 1, 1).to(device)
resnet_std  = torch.tensor([0.229, 0.224, 0.225]).view(3, 1, 1).to(device)

# 4. 特征提取函数
@torch.no_grad()
def prepare_and_extract(x_data, batch_size=500):
    # 【核心修复】将输入的 CPU 张量移入 GPU，与 device 上其他张量对齐
    x_data = x_data.to(device)

    # 逆归一化回 [0, 1]
    x_norm = (x_data * 0.5 + 0.5)
    # 应用 ImageNet 归一化
    x_norm = (x_norm - resnet_mean) / resnet_std

    features = []
    # 分批次提取
    for i in range(0, x_norm.size(0), batch_size):
        batch = x_norm[i:i+batch_size]
        features.append(model(batch))
    return torch.cat(features, dim=0)

# --- 执行提取 ---
print("正在提取 ResNet 特征，请稍候...")
X_train = prepare_and_extract(X_train)
X_test  = prepare_and_extract(X_test)

# 【重要】将标签数据也移至 GPU
# 否则后续计算 ytr[k_indices] 时会因设备不匹配（CPU vs GPU）报错
y_train = y_train.to(device)
y_test  = y_test.to(device)

print(f"X_train shape: {X_train.shape}")
print(f"X_test shape: {X_test.shape}")

Using device: cuda
正在提取 ResNet 特征，请稍候...
X_train shape: torch.Size([15000, 512])
X_test shape: torch.Size([35000, 512])


In [6]:
class NearestNeighbor(object):
  def __init__(self,k=3):
    self.k=k

  def train(self, X, y):
    """ X is N x D where each row is an example. Y is 1-dimension of size N """
    # the nearest neighbor classifier simply remembers all the training data
    self.Xtr = X
    self.ytr = y

  def predict(self, X):
    """ X is N x D where each row is an example we wish to predict label for """
    num_test = X.shape[0]
    # let's make sure that the output type matches the input type
    Ypred = torch.zeros(num_test, dtype = self.ytr.dtype)

    # loop over all test rows
    for i in range(num_test):
      # find the nearest training image to the i'th test image
      # using the L2 distance (ignore sqrt to save a little time)
      distances = torch.sum(torch.square((self.Xtr - X[i,:])), dim=1)
      _, k_indices = torch.topk(distances, self.k, dim=-1, largest=False, sorted=False)
      # ->get the index with k-smallest distance
      k_labels=self.ytr[k_indices]
      Ypred[i] = torch.bincount(k_labels).argmax() # predict the label of the nearest example
    return Ypred

  def predict_vec(self,X):
        # 1. 批量计算 L2 距离矩阵 [num_test, num_train]
        # torch.cdist 默认计算欧氏距离，会自动使用 GPU 并行计算
        distances = torch.cdist(X, self.Xtr)

        # 2. 获取每个测试样本距离最小的 k 个索引 [num_test, k]
        _, k_indices = torch.topk(distances, self.k, dim=-1, largest=False, sorted=False)

        # 3. 提取对应的训练标签 [num_test, k]
        k_labels = self.ytr[k_indices]

        # 4. 沿 k 维度投票，取众数 (mode) 作为预测类别 [num_test]
        Ypred, _ = torch.mode(k_labels, dim=-1)

        return Ypred


In [7]:
nn_5=NearestNeighbor(k=5)
nn_5.train(X_train, y_train)
nn_7=NearestNeighbor(k=7)
nn_7.train(X_train, y_train)

In [8]:
#y_pred=nn_5.predict(X_test)
#acc = (y_pred == y_test).float().mean()
#print ('accuracy: %f' % (acc,))

In [9]:
#y_pred_7=nn_7.predict(X_test)
#acc_ = (y_pred_7 == y_test).float().mean()
#print ('accuracy: %f' % (acc_,))

In [10]:
y_pred=nn_5.predict_vec(X_test)
acc = (y_pred == y_test).float().mean()
print ('accuracy: %f' % (acc,))

accuracy: 0.525400


In [11]:
y_pred_7=nn_7.predict_vec(X_test)
acc_ = (y_pred_7 == y_test).float().mean()
print ('accuracy: %f' % (acc_,))#从30%提升到54%

accuracy: 0.539086


In [12]:
nn_9=NearestNeighbor(k=9)
nn_9.train(X_train,y_train)
y_pred_9=nn_9.predict_vec(X_test)
acc_9 = (y_pred_9 == y_test).float().mean()
print ('accuracy: %f' % (acc_9,))

accuracy: 0.541371


In [14]:
for i in range(1,200,6):
    nn_i=NearestNeighbor(k=i)
    nn_i.train(X_train,y_train)
    y_pred_i=nn_i.predict_vec(X_test)
    acc_i = (y_pred_i == y_test).float().mean()
    print (f'when k={i},accuracy: %f' % (acc_i,))

when k=1,accuracy: 0.477714
when k=7,accuracy: 0.539086
when k=13,accuracy: 0.553343
when k=19,accuracy: 0.556886
when k=25,accuracy: 0.555371
when k=31,accuracy: 0.556400
when k=37,accuracy: 0.553943
when k=43,accuracy: 0.555371
when k=49,accuracy: 0.553171
when k=55,accuracy: 0.553886
when k=61,accuracy: 0.552114
when k=67,accuracy: 0.549629
when k=73,accuracy: 0.548257
when k=79,accuracy: 0.546486
when k=85,accuracy: 0.545657
when k=91,accuracy: 0.545029
when k=97,accuracy: 0.543686
when k=103,accuracy: 0.541286
when k=109,accuracy: 0.539286
when k=115,accuracy: 0.537829
when k=121,accuracy: 0.536829
when k=127,accuracy: 0.535829
when k=133,accuracy: 0.535029
when k=139,accuracy: 0.533286
when k=145,accuracy: 0.532343
when k=151,accuracy: 0.531286
when k=157,accuracy: 0.530371
when k=163,accuracy: 0.529057
when k=169,accuracy: 0.527743
when k=175,accuracy: 0.527971
when k=181,accuracy: 0.526514
when k=187,accuracy: 0.525629
when k=193,accuracy: 0.525800
when k=199,accuracy: 0.524686